In [ ]:
#Excel Extract
#FINAL SCRIPT ICICI
 
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from selenium.common.exceptions import NoSuchElementException
 
import base64, os
from pathlib import Path
import pandas as pd
from openpyxl import Workbook
from concurrent.futures import ThreadPoolExecutor, as_completed
 
 
 
def create_driver():
    # Chrome options
    chrome_options = Options()
    chrome_options.add_argument("--disable-gpu")
    chrome_options.page_load_strategy = "none"
    chrome_options.add_argument("--window-size=1920x1080")
   
    # chrome_options.add_argument("--headless")
   
    prefs = {
        "download.prompt_for_download": False,
        "download.directory_upgrade": True,
        "safebrowsing.enabled": True,
    }
   
    chrome_options.add_experimental_option("prefs", prefs)
    driver = webdriver.Chrome(options=chrome_options)
    return driver
 
 
def save_page_as_pdf(driver, output_path):
    pdf = driver.execute_cdp_cmd("Page.printToPDF", {
        "printBackground": True,
        "paperWidth": 8.27,     # A4
        "paperHeight": 11.69,
        "marginTop": 0.4,
        "marginBottom": 0.4,
        "marginLeft": 0.4,
        "marginRight": 0.4
    })
 
    pdf_bytes = base64.b64decode(pdf["data"])
    Path(output_path).write_bytes(pdf_bytes)
 
 
def hard_stop_loading(driver):
    driver.execute_script("""
        window.stop();
        window.setInterval = () => {};
        window.setTimeout = () => {};
    """)
 
def find_button_by_ids(driver, button_ids):
    for button_id in button_ids:
        try:
            button = WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.ID, button_id))
            )
            if button.is_displayed():
                return button
        except TimeoutException:
            continue
    return None
 
def find_button_by_css_selector(driver):
    buttons = driver.find_elements(By.CSS_SELECTOR, '[id^="showMore"]')
    for button in buttons:
        if button.is_displayed():
            return button
    return None
 
def find_button_by_xpath(driver):
    buttons = driver.find_elements(By.XPATH, '//button[contains(@id, "showMore")]')
    for button in buttons:
        if button.is_displayed():
            return button
    return None
 
# Function to extract data from a single link
def extract_data(driver,link, index):
   
    driver.get(link)
    print(f"Page loaded : {link}")
    time.sleep(5)
    hard_stop_loading(driver)
    print(f"Hard stop executed.")
   
    try:
        # Attempt to find and click the 'Show Details' button
        button_ids = ['showMore', 'showMore1', 'showMore2', 'showMore3', 'showMore4',
                      'showMore5', 'showMore6', 'showMore7']
        show_more_button = find_button_by_ids(driver, button_ids)
        if not show_more_button:
            show_more_button = find_button_by_css_selector(driver)
        if not show_more_button:
            show_more_button = find_button_by_xpath(driver)
       
        if show_more_button:
            driver.execute_script("arguments[0].scrollIntoView(true);", show_more_button)
            driver.execute_script("arguments[0].click();", show_more_button)
            print(f"'Show Details' button clicked for link: {link}")
        else:
            raise Exception("Show Details button not found or not clickable.")
       
    except Exception as e:
        print(f"Error clicking 'Show Details' for link {index + 1}: {link} - {e}")
        return [[link, None, None, None]]  # Return as a list of list for consistency
 
    try:
        # Extract AUM value
        aum_element = driver.find_element(By.XPATH, "//td[contains(text(), 'Million')]")
        aum_value = aum_element.text.strip()
 
        # Extract SFIN value
        sfin_element = driver.find_element(By.XPATH, "//h1[contains(@class, 'gcolor')]/span[@class='font-s']")
        sfin_value = sfin_element.text.strip()
 
        # Wait for any table to be present
        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.TAG_NAME, "table"))
        )
        tables = driver.find_elements(By.TAG_NAME, "table")
       
        all_table_data = []
        for table in tables:
            rows = table.find_elements(By.TAG_NAME, "tr")
            for row in rows:
                cols = row.find_elements(By.TAG_NAME, "td")
                row_data = [link, aum_value, sfin_value] + [col.text.strip() for col in cols]
                all_table_data.append(row_data)
        print(f"Data saved for table in link: {link}")
        return all_table_data
       
    except Exception as e:
        print(f"Error extracting data from link {index + 1}: {link} - {e}")
        return [[link, None, None, None]]
   
   
   
# Run the data extraction in parallel using ThreadPoolExecutor
# with ThreadPoolExecutor(max_workers=4) as executor:  # Adjust max_workers based on your system's capability
#     futures = [executor.submit(extract_data, row['Link'], index) for index, row in links_df.iterrows()]
   
#     for future in as_completed(futures):
#         result = future.result()
#         if result is not None:
#             for row_data in result:
#                 ws.append(row_data)
 
 
def runner():
   
    link_file = r"ICICISHEET.xlsx"
    links_df = pd.read_excel(link_file)
   
    # Prepare the output Excel file
    output_file = 'output.xlsx'
    wb = Workbook()
    ws = wb.active
    ws.title = "Scheme Data"
    ws.append(["Link", "AUM", "SFIN", "Table Data"])
   
   
    for index,rows in links_df.iterrows():
        link = rows['Link']
        pdf_name = rows['pdf_name']
       
       
        driver = create_driver()
        result = extract_data(driver, link, index)
        if result is not None:
            for row_data in result:
                ws.append(row_data)
        driver.quit()
        print(f"Data extraction completed for {index + 1}: {link}")
        # save_page_as_pdf(driver, os.path.join(folder_path, f"{pdf_name}.pdf"))
        driver.quit()
        if index == 2:
            break
       
       
    # Save the output Excel file
    wb.save(output_file)
   
    print("Data extraction completed successfully.")
   
 
if __name__ == "__main__":
   
    folder_path = r"selenium-scrape\INSR_PDF"
   
    runner()  